# Telecom Customer Churn Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a telecom customer will **churn** (cancel service) based on account information, service usage, and contract details. ~7,043 rows with ~27% churn rate.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a customer's account details, service subscriptions, and demographics, predict whether they will churn. Retention is far cheaper than acquisition.

## 5 · Why This Project Matters

- Customer churn costs telecom companies billions annually.
- Acquiring a new customer costs **5-25x more** than retaining one.
- Churn prediction enables **proactive retention offers** (discounts, upgrades).
- This is one of the most common real-world classification problems.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~7,043 |
| Features | ~20 (numeric + categorical) |
| Target | `Churn` (binary: Yes/No) |
| Class balance | ~73% No, ~27% Yes |
| Missing values | Few in TotalCharges |

## 7 · Dataset Source and License Notes

**Source**: Telco Customer Churn dataset (Kaggle/OpenML).
**License**: Public / educational.
**Note**: IBM sample dataset for telecommunications churn analysis.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "Churn"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '.'
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Telecom Customer Churn Prediction


## 11 · Dataset Download or Loading

In [4]:
from sklearn.datasets import fetch_openml

data = fetch_openml(data_id=42178, as_frame=True, parser='auto')
df = data.frame.copy()

# Encode target
if df['Churn'].dtype == 'object' or df['Churn'].dtype.name == 'category':
    df['Churn'] = (df['Churn'].astype(str).str.strip() == 'Yes').astype(int)
else:
    df['Churn'] = df['Churn'].astype(int)

# Fix TotalCharges (may be string with spaces)
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Drop customerID if present
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,'No phone service',DSL,No,Yes,No,No,No,No,Month-to-month,Yes,'Electronic check',29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,'One year',No,'Mailed check',56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,'Mailed check',53.85,108.15,1
3,Male,0,No,No,45,No,'No phone service',DSL,Yes,No,Yes,Yes,No,No,'One year',No,'Bank transfer (automatic)',42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,'Fiber optic',No,No,No,No,No,No,Month-to-month,Yes,'Electronic check',70.70,151.65,1


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (7043, 20)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 22

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Dtypes:
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object

Target 'Churn' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(num_cols[:3]):
    df.boxplot(column=col, by=TARGET, ax=axes[i])
    axes[i].set_title(f"{col} by Churn")
plt.suptitle("")
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

# Contract type vs churn
if 'Contract' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    ct = pd.crosstab(df['Contract'], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, color=['steelblue', 'coral'])
    ax.set_title("Churn Rate by Contract Type")
    ax.set_ylabel("Proportion")
    plt.tight_layout()
    plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 4, Categorical: 15


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Churn Distribution")
axes[0].set_xticklabels(['No (0)', 'Yes (1)'], rotation=0)

df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions")
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / df[TARGET].value_counts().iloc[1]:.2f}:1")

Class distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Imbalance ratio: 2.77:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (5634, 19), Test: (1409, 19)
Train target dist: {0: 4139, 1: 1495}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean[:10]}...")

Features (19): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup']...


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.7935
Precision: 0.7850
Recall   : 0.7935
F1       : 0.7877
ROC-AUC  : 0.8359


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
QuadraticDiscriminantAnalysis  0.762952           0.754120  0.819869  0.772852   0.796799  0.762952    0.021634
GaussianNB                     0.734564           0.744189  0.802767  0.748304   0.790052  0.734564    0.019664
BernoulliNB                    0.763662           0.725574  0.811243  0.769424   0.778694  0.763662    0.023338
NearestCentroid                0.698368           0.723820  0.807314  0.715589   0.778021  0.698368    0.021767
AdaBoostClassifier             0.793471           0.723665  0.842019  0.790933   0.789050  0.793471    0.247002
LGBMClassifier                 0.796309           0.713644  0.833382  0.790411   0.787777  0.796309    0.100106
LogisticRegression             0.794180           0.713049  0.835924  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.7935
F1       : 0.7835


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

# Add baseline and FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8055  F1: 0.7963


LightGBM  Acc: 0.7913  F1: 0.7863


XGBoost   Acc: 0.7828  F1: 0.7778


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost               0.8055  0.7963     0.7952  0.8055
Logistic Regression    0.7935  0.7877     0.7850  0.7935
LightGBM               0.7913  0.7863     0.7836  0.7913
FLAML                  0.7935  0.7835     0.7816  0.7935
XGBoost                0.7828  0.7778     0.7749  0.7828

Top 3: ['CatBoost', 'Logistic Regression', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  CatBoost
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.68      0.51      0.58       374

    accuracy                           0.81      1409
   macro avg       0.76      0.71      0.73      1409
weighted avg       0.80      0.81      0.80      1409


  Logistic Regression
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.53      0.58       374

    accuracy                           0.79      1409
   macro avg       0.74      0.71      0.72      1409
weighted avg       0.78      0.79      0.79      1409


  LightGBM
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.62      0.54      0.58       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")

# Errors by class
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: CatBoost
Error rate: 0.1945 (274 / 1409)

Errors by true class:
      errors  total  error_rate
true                           
0         92   1035    0.088889
1        182    374    0.486631


## 25 · Interpretation and Business Insight

- **Contract type** is the strongest predictor — month-to-month customers churn most.
- **Tenure** is inversely related to churn — long-term customers rarely leave.
- **MonthlyCharges** correlates with churn — high spenders on short contracts churn.
- **TotalCharges** reflects tenure × monthly charges (interaction).
- Internet service type and tech support availability matter.

## 26 · Limitations

1. Single telecom company — patterns may not generalize.
2. No temporal dimension — can't model churn timing.
3. No customer interaction data (calls, complaints, usage patterns).
4. ~27% churn rate — moderate imbalance.
5. Contract type may dominate, masking subtler signals.

## 27 · How to Improve This Project

1. Add customer interaction data (call center contacts, app usage).
2. Model churn timing with survival analysis.
3. Build a customer lifetime value (CLV) model alongside churn.
4. Apply SHAP for individual churn risk explanations.
5. Create retention offer optimization (cost-benefit analysis).

## 28 · Production Considerations

- Daily/weekly churn scoring integrated with CRM.
- Automated retention offer triggering for high-risk customers.
- A/B testing retention interventions.
- Model drift monitoring as market conditions change.

## 29 · Common Mistakes

1. Using accuracy instead of recall/F1 for churn detection.
2. Not considering the cost asymmetry (missing a churner vs false alarm).
3. Ignoring contract type as a near-perfect predictor.
4. Not segmenting customers before modeling.
5. Deploying without a retention action plan.

## 30 · Mini Challenge / Exercises

1. Calculate the cost of churn vs retention offer — what's the optimal threshold?
2. Build a model using only contract + tenure — how close to the full model?
3. Segment customers by contract type and build separate models.
4. Use SHAP to explain why a specific customer is predicted to churn.

## 31 · Final Summary / Key Takeaways

1. Telecom churn is a high-value, well-understood classification problem.
2. Contract type and tenure dominate prediction — month-to-month is highest risk.
3. Boosting models (CatBoost, XGBoost) typically achieve ~80% F1.
4. Recall on the churn class matters most for business impact.
5. Combine churn prediction with retention offer optimization.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }

with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.8055,
    "f1": 0.7963,
    "precision": 0.7952,
    "recall": 0.8055
  },
  "LightGBM": {
    "accuracy": 0.7913,
    "f1": 0.7863,
    "precision": 0.7836,
    "recall": 0.7913
  },
  "XGBoost": {
    "accuracy": 0.7828,
    "f1": 0.7778,
    "precision": 0.7749,
    "recall": 0.7828
  },
  "Logistic Regression": {
    "accuracy": 0.7935,
    "f1": 0.7877,
    "precision": 0.785,
    "recall": 0.7935
  },
  "FLAML": {
    "accuracy": 0.7935,
    "f1": 0.7835,
    "precision": 0.7816,
    "recall": 0.7935
  }
}
